# Phase 1.3: `probes/cv.py`, the leakage-safe split definition (tile 32UNU)

**Any number produced outside `probes/cv.py` does not exist.** This notebook is
where that module is exercised end to end: every runnable mode, every refusal,
the same-cube gate provoked on purpose, and the join contract to the Phase 1.2
embeddings asserted on real (cube, encoder) pairs.

**CPU is enough.** No GPU, no encoder weights, <= 12 GB. Nothing is fine-tuned
here and nothing is re-encoded.

**What must RAISE on this subset, and is not a bug.** The 20 cubes are one tile
(32UNU) and one year (2018). So `cube`, `spatial_block` and `temporal` run,
while `year`, `tile` and `crossed` all correctly refuse. Step 8 asserts the
refusals; a green Step 8 means the guards work.

**Drive layout: one SUBFOLDER per phase, under one project folder.** Deleting
a phase's subfolder removes everything that phase created and nothing another
phase depends on:

```
My Drive/
└── NeurIPS-CCAI-2026/
    ├── data/raw/*.nc         SHARED cubes. NOT a phase -- every phase reads
    │                         them, and reset_phase refuses to touch them.
    ├── phase1_1/             checkout + notebook
    ├── phase1_2/             checkout; artefacts at data/phase1_2/{embeddings,masks}
    └── phase1_3/             checkout
        └── phase1_3_repo.zip <- drag it here, leave it zipped
```

Step 2 finds the cubes and the Phase 1.2 artefacts wherever they sit on Drive
(searching up to three levels down) and reads them IN PLACE. It never writes
into another phase's subfolder.

The `phase1_2/data/phase1_2/` nesting is redundant but deliberate: the outer
name is the Drive checkout, the inner one is `data.paths.phase_dir`, which is
canonical and not worth bending for cosmetics.

## Step 1: Install, then restart

CPU only. No `satlaspretrain-models` and no model weights: this phase reads the
`.npz` files Phase 1.2 already wrote.

In [ ]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_3_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run the notebook against your own environment (pip install -r "
          "requirements.txt) and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    !pip install earthnet s3fs xarray zarr netCDF4

    # torch arrives with Colab and is imported transitively by encoders/.
    # Installing over Colab's build swaps in a slower wheel for no gain here.
    if importlib.util.find_spec("torch") is None:
        !pip install torch

    import subprocess, sys
    probe = "import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, torch"
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the manifest."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

Extracts `phase1_3_repo.zip` into **its own** `phase1_3/` subfolder, then
resolves two READ-ONLY inputs that live wherever they already are:

* `data/raw/` — the 20 cubes. Shared across phases and never cleared.
* `data/phase1_2/embeddings/` — the 100 `.npz` from Phase 1.2.

Everything Phase 1.3 writes goes under this subfolder's `data/phase1_3/`, so
`reset_phase("phase1_3")` (or deleting the `phase1_3/` subfolder) is a
complete undo.

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "probes/cv.py",
            "tests/test_cv_folds.py", "tests/conftest.py"]
ZIP_NAME = "phase1_3_repo.zip"
PHASE = "phase1_3"
INPUT_PHASE = "phase1_2"          # read-only: this phase never writes to it

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "cv.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.3 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_3
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_3/ removes
        everything Phase 1.3 created and nothing Phase 1.2 depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# Phase 1.2 may have run in a different Drive folder. Read its artefacts in
# place; copying 70 MB of cubes per phase is waste, and writing into its
# folder would break the "delete this folder to undo this phase" property.
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the Phase 1.3 checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- this phase's OWN outputs ----------------------------------------------
FOLDS = phase_dir(PHASE, "folds")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz, READ-ONLY)"
      + ("" if n_emb else "   <- MISSING, see Step 3"))
print(f"FOLDS   {FOLDS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

# The interpreter that imported this repo, not whatever `python` resolves to.
# On a pyenv/venv machine a bare `python` may not exist at all, and on Colab it
# may not be the kernel's interpreter -- either way a subprocess would then
# test different code than the notebook is holding.
PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Environment check

Phase 1.3 needs no GPU and no encoder weights. It does need the Phase 1.2
embeddings, because Step 10 asserts the join contract against them.

This cell also runs the cheap half of the cache audit, so a stale or duplicated
cache fails here in seconds rather than at Step 10, after the manifest has been
built from 20 cubes.

In [ ]:
import glob, os, shutil
import numpy as np, pandas as pd

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
try:
    import torch
    print("torch      ", torch.__version__,
          "| CPU is sufficient for this phase; no weights are loaded")
except ImportError:
    raise RuntimeError("torch missing: encoders/ imports it transitively. Re-run Step 1.")

n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\ncubes      {len(glob.glob(os.path.join(RAW, '*.nc')))} in {RAW}")
print(f"embeddings {n_emb} in {EMB_IN}")
if n_emb == 0:
    raise RuntimeError(textwrap.dedent(f"""
        No Phase 1.2 embeddings found. Step 10 cannot assert the join contract
        without them.

        Fix: run notebooks/phase1_2_encoders.ipynb (its own Drive folder is
        fine -- Step 2 searches Drive for data/{INPUT_PHASE}/embeddings and
        reads it in place), then re-run Step 2 here.

        Searched from: {REPO}  and one/two levels under {DRIVE}
    """).strip())

free = shutil.disk_usage(REPO).free / 1e9
print(f"free space {free:.1f} GB (this phase writes a few hundred kB of fold indices)")
assert free > 0.5, "less than 0.5 GB free, clear space on Drive first"

# --- fail fast on the cache -------------------------------------------------
# Step 10 is where the join contract is asserted, but it runs AFTER the manifest
# has been built from 20 cubes. Discovering a stale or duplicated cache there
# costs a minute for nothing, so the cheap half of the audit runs here.
# cube_ids is not known yet (no manifest), so foreign files are not classified;
# Step 10 does that.
from encoders.pipeline import SCHEMA_VERSION, audit_embeddings

print()
_pre = audit_embeddings(EMB_IN, cube_ids=None)
assert _pre.current, (
    f"no usable embedding in {EMB_IN}. Step 10 cannot assert the join contract "
    "against it. Read the [audit] lines above: each defect names its own "
    "remedy. If every line is empty, the path itself is wrong -- check the "
    "[resolve] lines in Step 2."
)
print()
print(f"cache usable: {len(_pre.current)} pair(s) at v{SCHEMA_VERSION}. Step 10 "
      "checks coverage")
print("against the manifest and asserts the join contract on every one of them.")

## Step 4: The cubes

The manifest is built from `data/raw/*.nc`, **not** from the `.npz` files.
Already-present cubes are skipped.

In [ ]:
if len(glob.glob(os.path.join(RAW, "*.nc"))) >= 20:
    print(f"20 cubes already in {RAW}, skipping the download")
else:
    sh(f"{PY} -m data.download_greenearthnet --out '{RAW}' --n 20 --tile 32UNU")
print(f"{len(glob.glob(os.path.join(RAW, '*.nc')))} cubes in {RAW}")

## Step 5: Unit tests

Expect **`261 passed, 5 skipped`** (266 collected). The 5 skips are the
weight-downloading and MI batch-invariance tests, gated behind
`PHASE1_2_WEIGHTS=1`.

A **different collected count** than you get locally means the bundle is stale:
`make_zip.sh` lists files with `git ls-files`, so an uncommitted file is
silently absent. Commit, rebuild, re-upload. That is a signal, not noise.

In [ ]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which SUPPRESSES the final "N passed, N skipped" line -- the exact
# number the runbook tells you to compare against, and the project's
# stale-bundle signal. Let the ini file own the verbosity.
sh(f"{PY} -m pytest tests")

## Step 6: The REAL manifest, built from the cubes

`encoders.manifest.build_manifest` already exists — it is imported, never
rebuilt. One row per RETAINED (cube, frame).

In [ ]:
import glob
from data.loader import load_cube
from encoders.manifest import assert_strata_present, build_manifest

paths = sorted(glob.glob(os.path.join(RAW, "*.nc")))
assert len(paths) == 20, f"expected 20 cubes, found {len(paths)}"
samples = [load_cube(p, verbose=False) for p in paths]
MANIFEST = build_manifest(samples)
assert_strata_present(MANIFEST)

print(f"\nmanifest {MANIFEST.shape}   rows x columns")
print(f"cubes  {MANIFEST.cube_id.nunique()}   tiles {sorted(MANIFEST.tile.unique())}   "
      f"years {sorted(MANIFEST.year.unique())}")
print(f"clear_frac  min {MANIFEST.clear_frac.min():.3f}  "
      f"median {MANIFEST.clear_frac.median():.3f}  max {MANIFEST.clear_frac.max():.3f}")
print(f"timestamps  {str(MANIFEST.timestamp.min())[:10]} .. "
      f"{str(MANIFEST.timestamp.max())[:10]}")
assert len(MANIFEST) == 264, f"expected 264 retained frames, got {len(MANIFEST)}"
print("\nONE TILE, ONE YEAR -- so year / tile / crossed MUST refuse (Step 8).")

## Step 7: The three runnable modes

`cube` (default), `spatial_block` and `temporal`. Each prints, per fold:
n_train rows, n_test rows, n_train cubes, n_test cubes, and the years on each
side. The clear-fraction filter runs before every split.

In [ ]:
from probes import cv

print("=" * 70, "\ncube, k=5   (DEFAULT: GroupKFold on cube_id)\n" + "=" * 70)
cube5 = list(cv.folds(MANIFEST, "cube", k=5))

print("\n" + "=" * 70)
print("leave-one-cube-out   (the honest choice at 20-cube scale)\n" + "=" * 70)
loco = list(cv.leave_one_cube_out(MANIFEST, verbose=False))
print(f"[cv] {len(loco)} folds, each holding out exactly one cube; "
      f"test sizes {sorted(te.size for _, te in loco)}")

print("\n" + "=" * 70)
print("spatial_block, k=5   (SUBSTITUTE for tile holdout)\n" + "=" * 70)
blocks = list(cv.folds(MANIFEST, "spatial_block", k=5))

print("\n" + "=" * 70)
print("temporal, cutoff 2018-08-15   (P3 robustness variant, NOT a default)\n"
      + "=" * 70)
temporal = list(cv.folds(MANIFEST, "temporal", cutoff="2018-08-15"))

# Independent re-check of the leakage rule, computed here rather than trusted.
for name, fs in [("cube", cube5), ("loco", loco), ("spatial_block", blocks),
                 ("temporal", temporal)]:
    for i, (tr, te) in enumerate(fs):
        a = set(MANIFEST.cube_id.to_numpy()[tr])
        b = set(MANIFEST.cube_id.to_numpy()[te])
        assert not (a & b), f"{name} fold {i}: cube on both sides"
        assert not np.intersect1d(tr, te).size
print(f"\nRE-CHECKED independently: no cube on both sides in any of "
      f"{len(cube5) + len(loco) + len(blocks) + len(temporal)} folds")

## Step 8: The three refusals

`year`, `tile` and `crossed` must RAISE here. A raise is the designed
behaviour on a single-tile, single-year subset — read the messages: each one
names the correct fallback and warns against a random split.

In [ ]:
expected = [
    ("year", lambda: list(cv.folds(MANIFEST, "year")), cv.SingleYearError),
    ("tile", lambda: list(cv.folds(MANIFEST, "tile")), cv.SingleTileError),
    ("crossed", lambda: list(cv.folds(MANIFEST, "crossed")), cv.SingleYearError),
]
for name, call, err in expected:
    try:
        call()
        raise AssertionError(f"{name} mode did NOT raise -- the guard is broken")
    except err as e:
        print(f"--- {name} mode raised {type(e).__name__}, correctly ---")
        print(str(e), "\n")
print("all three refusals fired")

## Step 9: The same-cube gate, provoked on purpose

The gate lives INSIDE the splitter, not in the caller. Two provocations on
small synthetic manifests:

1. A **seasonal-shaped** manifest where one cube spans 2018–2020. Year grouping
   must then split within a cube, so `year` mode refuses and names `crossed`.
   `crossed` handles the same manifest cleanly — that is the collision resolved.
2. A manifest with a **duplicated `(cube_id, timestamp)` row**, which must fail
   loudly rather than duplicate a frame across folds.

In [ ]:
def synthetic(cubes, years, frames=4):
    rows = []
    for c in cubes:
        i = 0
        for y in years:
            for f in range(frames):
                rows.append({"cube_id": c, "tile": "33TAN", "year": y,
                             "timestamp": np.datetime64(f"{y}-05-01")
                                          + np.timedelta64(10 * f, "D"),
                             "original_axis_index": i,
                             "pixel_bbox": (0, 128, 0, 128), "clear_frac": 0.8})
                i += 1
    return pd.DataFrame(rows)

seasonal = synthetic([f"S{c}.nc" for c in range(4)], [2018, 2019, 2020])
print(f"synthetic seasonal manifest: {len(seasonal)} rows, "
      f"{seasonal.cube_id.nunique()} cubes each spanning "
      f"{sorted(seasonal.year.unique())}\n")

try:
    list(cv.year_folds(seasonal, k=3, verbose=False))
    raise AssertionError("the same-cube gate did NOT fire")
except cv.LeakageError as e:
    print("--- year mode on a multi-year cube raised LeakageError, correctly ---")
    print(str(e), "\n")

print("--- crossed mode on the SAME manifest, which is the resolution ---")
for tr, te in cv.crossed_folds(seasonal, k=3):
    ts = np.asarray(seasonal.timestamp.to_numpy(), dtype="datetime64[ns]")
    yr = ts.astype("datetime64[Y]").astype(int) + 1970
    assert not (set(seasonal.cube_id.to_numpy()[tr]) & set(seasonal.cube_id.to_numpy()[te]))
    assert not (set(yr[tr]) & set(yr[te]))
print("crossed: every test row's cube AND year unseen in train (asserted)\n")

dup = pd.concat([seasonal, seasonal.iloc[[0]]], ignore_index=True)
try:
    list(cv.cube_folds(dup, verbose=False))
    raise AssertionError("duplicate rows did NOT fail")
except cv.LeakageError as e:
    print("--- a duplicated (cube_id, timestamp) row raised LeakageError ---")
    print(str(e))

## Step 10: The join contract, on EVERY real (cube, encoder) pair

    (cube_id, original_axis_index)  ==  (cube, kept_idx)

`encoders.pipeline.audit_embeddings` decides which files in the directory are
really ours — imported, never re-derived, so P1–P4 make the same decision this
notebook does. Three things accumulate in a shared Drive folder that look like
embeddings and are not: Google Drive duplicates (`Copy of …`, and its localised
forms), artefacts from an older schema, and files for cubes from another tile.

Then **coverage before contract**. Asserting the join on one pair proves the
contract; asserting it on all of them proves the cache is whole. A cube
silently missing one encoder turns a per-encoder comparison into a comparison
over *different cubes*, and no downstream assertion can detect that.

`window_span_days` is carried through and checked per encoder: exactly 0 for
single-image, varying for the multi-image control.

In [ ]:
from encoders.pipeline import (SCHEMA_VERSION, assert_embeddings_complete,
                               audit_embeddings, load_encoded)

# The audit is imported, not re-derived here. It is the one place that decides
# which files in a shared Drive folder are really ours, so P1-P4 make the same
# decision this notebook does. Passing cube_ids is what lets it separate a
# foreign cube from one of ours. Google Drive names a duplicate "Copy of
# <name>" and localises that prefix ("Kopie von", "Copie de", ...), so the
# prefix is never matched on: the filename is compared against the one derived
# from the file's OWN stored cube and encoder, which is exact in every language.
CUBE_IDS = set(MANIFEST.cube_id)
AUDIT = audit_embeddings(EMB_IN, cube_ids=CUBE_IDS)

# Coverage BEFORE contract. Asserting the join on one pair proves the contract;
# asserting it on all of them proves the cache is whole. A cube silently
# missing one encoder turns a per-encoder comparison into a comparison over
# different cubes, and no downstream assertion can detect that.
print()
assert_embeddings_complete(AUDIT, CUBE_IDS)

# Now the contract itself, on EVERY pair rather than one per encoder.
joined, n_rows = {}, 0
for (cube, enc), path in sorted(AUDIT.current.items()):
    ec = load_encoded(path)
    out = cv.join_embeddings(MANIFEST, ec, verbose=False)
    rows = out["manifest_idx"]
    # Re-assert the contract independently of the function that just claimed it.
    assert (MANIFEST.original_axis_index.to_numpy()[rows] == ec.kept_idx).all()
    assert (MANIFEST.cube_id.to_numpy()[rows] == cube).all()
    assert out["window_span_days"].shape[0] == rows.size == ec.embeddings.shape[0]
    joined.setdefault(enc, []).append(out)
    n_rows += rows.size

print()
print(f"JOIN CONTRACT asserted on {len(AUDIT.current)} (cube, encoder) pairs, "
      f"{n_rows} rows total")
print("  (cube_id, original_axis_index) == (cube, kept_idx), plus timestamps "
      "and clear_frac")

for enc in sorted(joined):
    outs = joined[enc]
    w = np.concatenate([o["window_span_days"] for o in outs])
    D = outs[0]["embeddings"].shape[1]
    rows_here = sum(o["manifest_idx"].size for o in outs)
    assert rows_here == len(MANIFEST), (
        f"{enc}: joined {rows_here} rows but the manifest has {len(MANIFEST)}"
    )
    print(f"  {enc:<24} D={D:<5} {len(outs):>3} cubes  {rows_here:>4} rows  "
          f"window_span_days min {w.min():.0f} median {np.median(w):.0f} "
          f"max {w.max():.0f} d")

# Single-image encoders have no lookback by construction; the multi-image
# control does, and a CONSTANT one would mean the covariate carries no
# information -- i.e. it was never really cached.
mi = [e for e in joined if e.endswith("_mi_rgb")]
si = [e for e in joined if not e.endswith("_mi_rgb")]
for e in si:
    w = np.concatenate([o["window_span_days"] for o in joined[e]])
    assert (w == 0).all(), f"{e} is single-image; window_span_days must be 0"
if mi:
    w = np.concatenate([o["window_span_days"] for o in joined[mi[0]]])
    assert w.max() > 0, (
        f"{mi[0]} is multi-image but window_span_days is constant 0 -- the "
        "covariate P2/P3 must condition on was not cached"
    )
    print()
    print(f"{mi[0]}: lookback varies 0..{w.max():.0f} days over {w.size} rows --")
    print("weather-correlated, and carried through by cv.join_embeddings rather")
    print("than silently dropped.")
print(f"single-image encoders {si}: window_span_days == 0 exactly")
print("  (an honest constant, not a NaN)")

## Step 11: Save this phase's fold indices

Written through `data/paths.phase_dir`, never a hand-typed path. Everything
lands under this folder's `data/phase1_3/`, so `reset_phase("phase1_3")` — or
deleting the Drive folder — is a complete, phase-scoped undo.

In [ ]:
from data.paths import describe_phase, phase_dir

FOLDS = phase_dir(PHASE, "folds")
saved = []
for name, fs in [("cube_k5", cube5), ("loco", loco),
                 ("spatial_block_k5", blocks), ("temporal_2018-08-15", temporal)]:
    path = os.path.join(FOLDS, f"{name}.npz")
    np.savez_compressed(
        path,
        n_folds=np.array(len(fs)),
        n_manifest_rows=np.array(len(MANIFEST)),
        **{f"train_{i}": tr for i, (tr, _) in enumerate(fs)},
        **{f"test_{i}": te for i, (_, te) in enumerate(fs)},
    )
    saved.append(path)
    print(f"[phase1_3] {os.path.basename(path)}: {len(fs)} folds, "
          f"{os.path.getsize(path) / 1e3:.1f} kB")

# The manifest itself, so a probe can reproduce a fold without re-reading cubes.
mpath = os.path.join(FOLDS, "manifest.csv")
MANIFEST.to_csv(mpath, index=False)
print(f"[phase1_3] manifest.csv: {MANIFEST.shape} "
      f"({os.path.getsize(mpath) / 1e3:.0f} kB)")

# Read one back and re-assert it, rather than trusting the write.
z = np.load(saved[0])
assert int(z["n_manifest_rows"]) == len(MANIFEST)
tr0, te0 = z["train_0"], z["test_0"]
assert not set(MANIFEST.cube_id.to_numpy()[tr0]) & set(MANIFEST.cube_id.to_numpy()[te0])
print(f"\nre-read {os.path.basename(saved[0])}: fold 0 train {tr0.shape} "
      f"test {te0.shape}, still cube-disjoint")
describe_phase(PHASE)

## Step 12: Full listing from the absolute project root

Every subfolder and every file, no depth limit and no summarising -- the "did
everything land where I think it did" check, now including what Step 11 just
wrote.

It lists the **project root**, not this checkout: if this phase runs from
`NeurIPS-CCAI-2026/phase1_3/`, the root is its parent, so you see the shared
cubes and Phase 1.2's artefacts alongside Phase 1.3's own outputs. Standalone
-- run it on its own at any time.

In [ ]:
import os

# Standalone: re-resolve the project root if this cell is run on its own.
# REPO is this phase's checkout; the PROJECT root is its parent when the
# checkout sits in a phase subfolder, and REPO itself in the older flat layout.
if "REPO" not in globals():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _D = "/content/drive/MyDrive"
    except ImportError:
        _D = os.path.expanduser("~")
    REPO = f"{_D}/NeurIPS-CCAI-2026/phase1_3"

NESTED = os.path.basename(os.path.abspath(REPO)).startswith("phase1_")
PROJECT = os.path.dirname(os.path.abspath(REPO)) if NESTED else os.path.abspath(REPO)
assert os.path.isdir(PROJECT), f"not a directory: {PROJECT}"
print(f"checkout {os.path.abspath(REPO)}")
print(f"project  {PROJECT}   ({'nested' if NESTED else 'flat'} layout)\n")

HIDE = {".git", "__pycache__", ".venv", ".pytest_cache", ".ipynb_checkpoints",
        ".DS_Store", ".mypy_cache", "node_modules"}


def _h(n):
    for u in ("B", "kB", "MB", "GB"):
        if n < 1000 or u == "GB":
            return f"{n:.0f} {u}" if u == "B" else f"{n:.1f} {u}"
        n /= 1000.0


def _stats(path):
    n = b = 0
    for dp, dn, fn in os.walk(path):
        dn[:] = [d for d in dn if d not in HIDE]
        for f in fn:
            if f in HIDE:
                continue
            n += 1
            try:
                b += os.path.getsize(os.path.join(dp, f))
            except OSError:
                pass
    return n, b


LINES, N_FILES, N_DIRS, N_BYTES = [], 0, 0, 0


def tree(path, prefix=""):
    """Every entry, unlimited depth. Directories first, then files."""
    global N_FILES, N_DIRS, N_BYTES
    try:
        names = sorted(n for n in os.listdir(path) if n not in HIDE)
    except OSError as e:
        LINES.append(f"{prefix}<unreadable: {e}>")
        return
    dirs = [n for n in names if os.path.isdir(os.path.join(path, n))]
    files = [n for n in names if not os.path.isdir(os.path.join(path, n))]
    for i, name in enumerate(dirs + files):
        full = os.path.join(path, name)
        last = i == len(dirs) + len(files) - 1
        branch = "`-- " if last else "|-- "
        if name in dirs:
            n, b = _stats(full)
            N_DIRS += 1
            LINES.append(f"{prefix}{branch}{name}/{'':<{max(0, 44 - len(prefix) - len(name))}}"
                         f"[{n} files, {_h(b)}]")
            tree(full, prefix + ("    " if last else "|   "))
        else:
            sz = os.path.getsize(full)
            N_FILES += 1
            N_BYTES += sz
            LINES.append(f"{prefix}{branch}{name}{'':<{max(1, 45 - len(prefix) - len(name))}}"
                         f"{_h(sz)}")


LINES.append(f"{PROJECT}/")
tree(PROJECT)
print("\n".join(LINES))

print("\n" + "=" * 70)
print(f"TOTAL  {N_DIRS} folders, {N_FILES} files, {_h(N_BYTES)}")

print("\nBY TOP-LEVEL ENTRY")
for name in sorted(os.listdir(PROJECT)):
    if name in HIDE:
        continue
    full = os.path.join(PROJECT, name)
    if os.path.isdir(full):
        n, b = _stats(full)
        print(f"  {name + '/':<24} {n:>6} files  {_h(b):>10}")
    else:
        print(f"  {name:<24} {1:>6} file   {_h(os.path.getsize(full)):>10}")

print("\nDATA FILES BY LOCATION")
counts = {}
for dp, dn, fn in os.walk(PROJECT):
    dn[:] = [d for d in dn if d not in HIDE]
    for f in fn:
        if f.endswith((".nc", ".npz", ".csv")):
            rel = os.path.relpath(dp, PROJECT).replace(os.sep, "/")
            key = (rel, os.path.splitext(f)[1])
            counts[key] = counts.get(key, 0) + 1
for (rel, ext), n in sorted(counts.items()):
    print(f"  {rel + '/':<52} {n:>4} x {ext}")
if not counts:
    print("  none found -- if you expected cubes or embeddings, STOP and check.")

# --- what THIS phase produced, and what it only read -----------------------
print("\n" + "=" * 70)
print("PHASE 1.3 OUTPUTS")
folds = os.path.join(os.path.abspath(REPO), "data", "phase1_3", "folds")
if os.path.isdir(folds):
    got = sorted(f for f in os.listdir(folds) if f not in HIDE)
    print(f"  OK    {os.path.relpath(folds, PROJECT)}/  ({len(got)} files)")
    for f in got:
        print(f"          {f:<28} {_h(os.path.getsize(os.path.join(folds, f)))}")
    missing = [f for f in ("cube_k5.npz", "loco.npz", "spatial_block_k5.npz",
                           "manifest.csv") if f not in got]
    if missing:
        print(f"  WARN  expected but absent: {missing} -- re-run Step 11")
else:
    print(f"  TODO  no {os.path.relpath(folds, PROJECT)}/ yet -- run Step 11")

print("\nREAD-ONLY INPUTS (this phase never writes to these)")
for label, path in (("cubes", RAW if "RAW" in globals() else None),
                    ("phase1_2 embeddings", EMB_IN if "EMB_IN" in globals() else None)):
    if path is None:
        print(f"  {label:<22} (not resolved -- run Step 2)")
        continue
    inside = os.path.abspath(path).startswith(os.path.abspath(REPO) + os.sep)
    n = len([f for f in os.listdir(path)]) if os.path.isdir(path) else 0
    print(f"  {label:<22} {os.path.relpath(os.path.abspath(path), PROJECT)}  "
          f"({n} files){'  [inside this checkout]' if inside else ''}")

# --- layout check ----------------------------------------------------------
print("\n" + "=" * 70)
print("LAYOUT CHECK")
root = sorted(n for n in os.listdir(PROJECT) if n not in HIDE)
data_kids = (sorted(n for n in os.listdir(os.path.join(PROJECT, "data"))
                    if n not in HIDE) if "data" in root else [])
phases = [n for n in root if n.startswith("phase1_")
          and os.path.isdir(os.path.join(PROJECT, n))]
ok = True

if "data" in root and data_kids == ["raw"]:
    n_nc = len([f for f in os.listdir(os.path.join(PROJECT, "data", "raw"))
                if f.endswith(".nc")])
    print(f"  OK    data/raw/ present, {n_nc} cubes, and data/ holds nothing else")
elif "data" in root and "raw" in data_kids:
    print(f"  NOTE  data/ also holds {[k for k in data_kids if k != 'raw']} "
          "-- expected in the flat layout")
else:
    ok = False
    print("  WARN  no data/raw at the project root. The cubes are SHARED and "
          "belong here,\n        not inside a phase folder. Check where they went.")

if NESTED:
    stray = [n for n in root if n != "data" and not n.startswith("phase1_")]
    print(f"  {'OK   ' if phases else 'TODO '} phase folders: {phases or 'none yet'}")
    if stray:
        ok = False
        print(f"  TODO  {len(stray)} entr(y/ies) still loose at the root:"
              f"\n        {stray}\n        Run notebooks/organise_drive.ipynb "
              "to file them.")
    else:
        print("  OK    nothing loose at the root")
    print("\n" + ("Layout matches the target." if ok and phases else
          "Not in the target layout yet -- see notebooks/organise_drive.ipynb."))
else:
    print("  NOTE  FLAT layout: this checkout IS the project root, so the code "
          "sitting\n        beside data/ is expected, not stray. Everything above "
          "still ran.")
    print("        To move to one subfolder per phase, run "
          "notebooks/organise_drive.ipynb.")

out = os.path.join(FOLDS, "tree.txt") if "FOLDS" in globals() else "tree.txt"
with open(out, "w") as fh:
    fh.write("\n".join(LINES) + "\n")
print(f"\nfull listing written to {out} ({len(LINES)} lines)")

## Phase 1.3 is done when

```
Step 5   261 passed, 5 skipped   (266 collected)
Step 6   manifest (264, 21), 20 cubes, tile ['32UNU'], years [2018]
Step 7   cube k=5, LOCO (20 folds), spatial_block k=5, temporal all yield folds
         and the independent re-check finds no cube on both sides
Step 8   year -> SingleYearError, tile -> SingleTileError,
         crossed -> SingleYearError, each naming its fallback
Step 9   year mode on a multi-year cube -> LeakageError naming crossed;
         crossed handles the same manifest; a duplicate row -> LeakageError
Step 10  cache audited, coverage asserted, join contract on EVERY pair,
         window_span_days carried through (0 for single-image, 0-105 for MI)
Step 11  fold indices + manifest under data/phase1_3/folds/
Step 12  full recursive listing; PHASE 1.3 OUTPUTS all OK; LAYOUT CHECK
```

**Step 12 is the "did it land where I think it did" check.** It lists every
folder and file from the absolute PROJECT root -- the parent of this checkout
in the nested layout, so you see the shared cubes and Phase 1.2's artefacts
alongside what Step 11 just wrote. It also confirms the read-only inputs were
read from OUTSIDE this checkout: in the nested layout neither `cubes` nor
`phase1_2 embeddings` is tagged `[inside this checkout]`, which is what proves
this phase never wrote into another phase's folder. It is standalone -- run it
alone at any time.

**Everything this phase wrote is under `data/phase1_3/`.** To re-run cleanly:

```python
from data.paths import reset_phase
reset_phase("phase1_3")     # clears ONLY this phase; data/raw is untouched
```

Deleting the `phase1_3/` subfolder is the coarser version of the same undo,
and it cannot touch Phase 1.2's artefacts or the shared cubes, because this
notebook only ever READS them.

Next: P1/P2/P3/P4 import `probes.cv.folds` and nothing else. Any number that
does not come out of a fold from this module does not exist.